# Working with ERA5-Land Data using Python
এই notebook demonstrates how to download, read, explore, and crop **ERA5-Land climate data** using Python.

ERA5-Land is a reanalysis dataset provided by the Copernicus Climate Data Store (CDS). It contains land-surface variables such as soil moisture, temperature, runoff, etc., which are useful for hydrology, flood modeling, and environmental analysis.

## 1. Install Required Libraries
The following libraries are required:
- **cdsapi** → To download ERA5-Land data from Copernicus
- **xarray** → To read and manipulate multidimensional climate datasets
- **cfgrib** → To read GRIB formatted climate files
- **numpy** → For numerical operations

In [1]:
!pip install cdsapi xarray cfgrib numpy

## 2. Import Required Libraries
Here we import the necessary libraries for downloading and processing ERA5-Land data.

In [2]:
import cdsapi
import xarray as xr
import numpy as np

## 3. Download ERA5-Land Data from Copernicus
This code downloads ERA5-Land data using the **Copernicus Climate Data Store API (cdsapi)**.

Before running this cell:
1. Create an account at https://cds.climate.copernicus.eu
2. Configure your `.cdsapirc` file with your API key.

In [ ]:
# Initialize the CDS API client
c = cdsapi.Client(url="https://cds.climate.copernicus.eu/api", key="bffd2c4a-0ebb-40a5-b6d0-086c30ba9bec")

# Specify the dataset name
dataset = "reanalysis-era5-land"


# Request parameters
# Downloading a single variable (2m_temperature) for a specific date and time in grib format.
#  You can modify the parameters to download different variables, dates, or formats as needed.
request = {
    "variable": ["2m_temperature"],
    "year": "2025",
    "month": "08",
    "day": "11",
    "time": ["00:00"],
    "format": "grib"
}

# Download the dataset
c.retrieve(dataset, request, "era5_land_sample.grib")

2026-03-15 09:11:03,553 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land-timeseries?tab=overview)
2026-03-15 09:11:03,553 INFO Request ID is a4afc900-c007-437b-88ee-ef1b1f721502
2026-03-15 09:11:03,769 INFO status has been updated to accepted
2026-03-15 09:11:18,028 INFO status has been updated to successful


'era5_land_sample.grib'

## 4. Reading ERA5-Land Data using Xarray
ERA5-Land data is usually downloaded in **GRIB format**.
We use **xarray + cfgrib** backend to open the dataset.
(Here actually a different file is opened which actually has more variables then the recently downloaded once in order to give more idea about the file)

In [6]:
ds = xr.open_dataset( r"F:\ERA5-land\6effba2acabbaf6491fbb328c7bb4144\lsm_0.area-subset.22.72679.88.530579.22.369126.88.162537.nc"
)
ds

<xarray.Dataset> Size: 160B
Dimensions:     (latitude: 4, longitude: 4)
Coordinates:
  * latitude    (latitude) float64 32B 22.7 22.6 22.5 22.4
  * longitude   (longitude) float64 32B 88.2 88.3 88.4 88.5
    time        datetime64[ns] 8B ...
    step        timedelta64[ns] 8B ...
    surface     float64 8B ...
    valid_time  datetime64[ns] 8B ...
Data variables:
    lsm         (latitude, longitude) float32 64B ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-27T09:54 GRIB to CDM+CF via cfgrib-0.9.1...

## 5. Inspect Dataset Metadata
This shows information about:
- Dimensions
- Coordinates (time, latitude, longitude)
- Variables present in the dataset

In [7]:
print(ds)

<xarray.Dataset> Size: 160B
Dimensions:     (latitude: 4, longitude: 4)
Coordinates:
  * latitude    (latitude) float64 32B 22.7 22.6 22.5 22.4
  * longitude   (longitude) float64 32B 88.2 88.3 88.4 88.5
    time        datetime64[ns] 8B ...
    step        timedelta64[ns] 8B ...
    surface     float64 8B ...
    valid_time  datetime64[ns] 8B ...
Data variables:
    lsm         (latitude, longitude) float32 64B ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-27T09:54 GRIB to CDM+CF via cfgrib-0.9.1...


## 6. List All Variables in the Dataset
This helps identify the available variables such as soil moisture, temperature, etc.

In [8]:
list(ds.data_vars)

['lsm']

## 7. Access a Specific Variable
We can extract a particular variable from the dataset.
For example: surface temperature or soil moisture.

In [9]:
variable_data = ds[list(ds.data_vars)[0]]
variable_data

<xarray.DataArray 'lsm' (latitude: 4, longitude: 4)> Size: 64B
[16 values with dtype=float32]
Coordinates:
  * latitude    (latitude) float64 32B 22.7 22.6 22.5 22.4
  * longitude   (longitude) float64 32B 88.2 88.3 88.4 88.5
    time        datetime64[ns] 8B ...
    step        timedelta64[ns] 8B ...
    surface     float64 8B ...
    valid_time  datetime64[ns] 8B ...
Attributes: (12/30)
    GRIB_paramId:                             172
    GRIB_dataType:                            an
    GRIB_numberOfPoints:                      6483600
    GRIB_typeOfLevel:                         surface
    GRIB_stepUnits:                           1
    GRIB_stepType:                            instant
    ...                                       ...
    GRIB_name:                                Land-sea mask
    GRIB_shortName:                           lsm
    GRIB_units:                               (0 - 1)
    long_name:                                Land-sea mask
    units:                                    (0 - 1)
    standard_name:                            land_binary_mask

## 8. Access Raw Values of the Variable
The `.values` attribute returns the numerical array stored in the dataset.

In [10]:
variable_data.values

array([[1.        , 0.9868312 , 0.9614067 , 0.9804027 ],
       [0.9578643 , 0.93145347, 0.88961947, 0.9462017 ],
       [0.81910074, 0.98707426, 0.90771246, 0.9723669 ],
       [0.9556434 , 0.9991065 , 0.999284  , 0.9999864 ]], dtype=float32)

## 9. Inspect Time Information
ERA5-Land stores timestamps in **numpy datetime64 format**.

In [11]:
ds.time.values

np.datetime64('2013-11-29T00:00:00.000000000')

## 10. Crop the Dataset for a Specific Region
We can extract a **smaller geographic region** using latitude and longitude slicing.

This is useful when working with a specific study area for flood mapping.

In [12]:
cropped = ds.sel(
    latitude=slice(22.6, 22.4),
    longitude=slice(88.2, 88.4)
)

cropped

<xarray.Dataset> Size: 80B
Dimensions:     (latitude: 2, longitude: 2)
Coordinates:
  * latitude    (latitude) float64 16B 22.5 22.4
  * longitude   (longitude) float64 16B 88.3 88.4
    time        datetime64[ns] 8B 2013-11-29
    step        timedelta64[ns] 8B ...
    surface     float64 8B ...
    valid_time  datetime64[ns] 8B ...
Data variables:
    lsm         (latitude, longitude) float32 16B 0.9871 0.9077 0.9991 0.9993
Attributes:
    GRIB_edition:            2
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-27T09:54 GRIB to CDM+CF via cfgrib-0.9.1...

## 11. Check Coordinates of Cropped Region

In [13]:
cropped.latitude.values

array([22.5, 22.4])

## 12. Opening Grib file using cfgrib

In [17]:
#OPEN THE GRIB FILE USING CFGRIB, As the data is in grib format, we can use the python package cfgrib to read the data. 
# The following code is used to read the data and print the variables in the dataset.
import cfgrib

dataset = cfgrib.open_datasets(
    r"F:\ERA5-land\6effba2acabbaf6491fbb328c7bb4144\data.grib"
)
dataset

c:\Users\Admin\miniconda3\envs\Era5land_down\Lib\site-packages\cfgrib\xarray_store.py:51: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  o = xr.merge([o, ds], **kwargs)
c:\Users\Admin\miniconda3\envs\Era5land_down\Lib\site-packages\cfgrib\xarray_store.py:51: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  o = xr.merge([o, ds], **kwargs)


[<xarray.Dataset> Size: 14kB
 Dimensions:              (time: 96, latitude: 4, longitude: 4)
 Coordinates:
   * time                 (time) datetime64[ns] 768B 2025-08-11 ... 2025-08-21...
     valid_time           (time) datetime64[ns] 768B 2025-08-11 ... 2025-08-21...
   * latitude             (latitude) float64 32B 22.67 22.57 22.47 22.37
   * longitude            (longitude) float64 32B 88.16 88.26 88.36 88.46
     number               int64 8B 0
     step                 timedelta64[ns] 8B 00:00:00
     depthBelowLandLayer  float64 8B 0.0
 Data variables:
     swvl1                (time, latitude, longitude) float32 6kB ...
     stl1                 (time, latitude, longitude) float32 6kB ...
 Attributes:
     GRIB_edition:            1
     GRIB_centre:             ecmf
     GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
     GRIB_subCentre:          0
     Conventions:             CF-1.7
     institution:             European Centre for Medium-Range 

## # The GRIB file may contain multiple datasets.
This command prints the number of datasets present inside the GRIB file.

In [18]:
print(len(dataset))

7


 Iterate through each dataset in the GRIB file.
data_vars contains all the variables present in that dataset.

For each dataset we print:
 1. Number of variables present
 2. Names of the variables

In [19]:
for i in dataset:
    print( len(i.data_vars), i.data_vars.keys())

2 KeysView(Data variables:
    swvl1    (time, latitude, longitude) float32 6kB ...
    stl1     (time, latitude, longitude) float32 6kB ...)
2 KeysView(Data variables:
    swvl2    (time, latitude, longitude) float32 6kB ...
    stl2     (time, latitude, longitude) float32 6kB ...)
2 KeysView(Data variables:
    swvl3    (time, latitude, longitude) float32 6kB ...
    stl3     (time, latitude, longitude) float32 6kB ...)
2 KeysView(Data variables:
    swvl4    (time, latitude, longitude) float32 6kB ...
    stl4     (time, latitude, longitude) float32 6kB ...)
21 KeysView(Data variables:
    sro      (time, step, latitude, longitude) float32 9kB ...
    ssro     (time, step, latitude, longitude) float32 9kB ...
    es       (time, step, latitude, longitude) float32 9kB ...
    lai_lv   (time, step, latitude, longitude) float32 9kB ...
    lai_hv   (time, step, latitude, longitude) float32 9kB ...
    sp       (time, step, latitude, longitude) float32 9kB ...
    sshf     (time, step, 

# Access a specific variable from the dataset.

In [20]:
dataset[0]["swvl1"].values

array([[[0.39088857, 0.39213026, 0.39325178, 0.39246023],
        [0.4061588 , 0.4074844 , 0.40666044, 0.40585172],
        [0.4504627 , 0.44965208, 0.4482826 , 0.4611324 ],
        [0.46415746, 0.4627918 , 0.48032224, 0.48142278]],

       [[0.3903883 , 0.39160138, 0.39270574, 0.39193708],
        [0.40555173, 0.40685254, 0.40602285, 0.4052294 ],
        [0.44979268, 0.4489172 , 0.44752485, 0.4605158 ],
        [0.46344548, 0.46201688, 0.47955877, 0.48074895]],

       [[0.38966835, 0.390952  , 0.39211738, 0.3914441 ],
        [0.4048394 , 0.4062146 , 0.40542495, 0.40471733],
        [0.44908988, 0.4482621 , 0.44687164, 0.4599446 ],
        [0.4627217 , 0.46135604, 0.47892272, 0.48015678]],

       ...,

       [[0.43045288, 0.42945534, 0.42898422, 0.42519814],
        [0.43782288, 0.4357229 , 0.43467957, 0.42774636],
        [0.47226006, 0.44549614, 0.4445806 , 0.47588402],
        [0.45242745, 0.4120832 , 0.44380623, 0.48043305]],

       [[0.43138897, 0.43141663, 0.4320594 , 0.4293

Each ERA5-Land variable contains metadata attributes.
One important attribute is 'long_name', which describes the variable.

This loop prints:
variable name → descriptive meaning

In [21]:
for ds in dataset:
    for var in ds.data_vars:
        print(var, ":", ds[var].attrs.get("long_name"))

swvl1 : Volumetric soil water layer 1
stl1 : Soil temperature level 1
swvl2 : Volumetric soil water layer 2
stl2 : Soil temperature level 2
swvl3 : Volumetric soil water layer 3
stl3 : Soil temperature level 3
swvl4 : Volumetric soil water layer 4
stl4 : Soil temperature level 4
sro : Surface runoff
ssro : Sub-surface runoff
es : Snow evaporation
lai_lv : Leaf area index, low vegetation
lai_hv : Leaf area index, high vegetation
sp : Surface pressure
sshf : Time-integrated surface sensible heat net flux
slhf : Time-integrated surface latent heat net flux
u10 : 10 metre U wind component
v10 : 10 metre V wind component
t2m : 2 metre temperature
d2m : 2 metre dewpoint temperature
ssrd : Surface short-wave (solar) radiation downwards
strd : Surface long-wave (thermal) radiation downwards
ssr : Surface net short-wave (solar) radiation
str : Surface net long-wave (thermal) radiation
e : Evaporation
ro : Runoff
tp : Total precipitation
fal : Forecast albedo
pev : Potential evaporation
src : Sk

# ERA5-Land datasets contain a time dimension.
This command retrieves the timestamps associated with the data.

In [22]:
ds['time'].values

array(['2025-08-10T00:00:00.000000000', '2025-08-11T00:00:00.000000000',
       '2025-08-12T00:00:00.000000000', '2025-08-19T00:00:00.000000000',
       '2025-08-20T00:00:00.000000000', '2025-08-21T00:00:00.000000000'],
      dtype='datetime64[ns]')

# Extract a smaller geographic region from the global dataset.
ds.sel() allows us to select a subset of the dataset
based on latitude and longitude coordinates.

In [23]:
cropped = ds.sel(
    latitude=slice(22.6, 22.4),
    longitude=slice(88.2, 88.4)
)


# Display the latitude values present in the cropped dataset.

In [24]:
cropped.latitude.values

array([22.569127, 22.469127])

# Inspect the cropped dataset.

cropped["time"].values → shows the timestamps of the cropped data.

cropped["evatc"].values → extracts the numerical values of the variable 'evatc'
(Evaporation from canopy or related evaporation variable depending on dataset).

 This returns the actual grid values for the selected region.

In [25]:
cropped["time"].values
cropped["evatc"].values

array([[[[            nan,             nan],
         [            nan,             nan]],

        [[            nan,             nan],
         [            nan,             nan]],

        [[            nan,             nan],
         [            nan,             nan]],

        [[            nan,             nan],
         [            nan,             nan]],

        [[            nan,             nan],
         [            nan,             nan]],

        [[            nan,             nan],
         [            nan,             nan]],

        [[            nan,             nan],
         [            nan,             nan]],

        [[            nan,             nan],
         [            nan,             nan]],

        [[            nan,             nan],
         [            nan,             nan]],

        [[            nan,             nan],
         [            nan,             nan]],

        [[            nan,             nan],
         [            nan,         

# Check the shape (dimensions) of the cropped variable.

In [26]:
cropped['evatc'].shape

(6, 24, 2, 2)

# Select data corresponding to a single timestamp

In [27]:
single_time = ds.sel(time="2025-08-11T00:00:00")


Extract the numerical values of the variable 'evatc'
for the selected timestamp.

In [28]:
single_time['evatc'].values

array([[[-7.07889103e-06, -8.61556509e-06, -8.54069549e-06,
         -5.94755875e-06],
        [-8.21580943e-06, -9.59343470e-06, -9.29323596e-06,
         -6.74203511e-06],
        [-7.57657654e-06, -8.66522714e-06, -8.74311445e-06,
         -7.12614383e-06],
        [-6.89540775e-06, -8.04796764e-06, -8.19132856e-06,
         -6.95387007e-06]],

       [[-2.22155541e-05, -2.76680930e-05, -2.74372287e-05,
         -1.88415324e-05],
        [-2.66620264e-05, -3.18827661e-05, -3.18975581e-05,
         -2.26843076e-05],
        [-2.56773292e-05, -2.92918940e-05, -3.01303153e-05,
         -2.45112205e-05],
        [-2.30036367e-05, -2.66792413e-05, -2.76911614e-05,
         -2.39475667e-05]],

       [[-3.83219740e-05, -5.27817028e-05, -5.66472227e-05,
         -4.80209965e-05],
        [-5.17215158e-05, -6.68816792e-05, -7.11802277e-05,
         -5.96705686e-05],
        [-5.61981724e-05, -6.75507763e-05, -7.04080448e-05,
         -6.07202855e-05],
        [-5.07426957e-05, -6.30005234e-

## Conclusion
In this notebook we demonstrated:

- How to download ERA5-Land data using **cdsapi**
- How to read **GRIB files using xarray and cfgrib**
- How to inspect dataset metadata
- How to extract variables and values
- How to crop ERA5-Land data for a specific region
- How to select data for a particular time step

This workflow forms the foundation for using ERA5-Land variables in applications such as **flood prediction, hydrological modeling, and climate analysis**.